# Testar voz neural treinada no Google Colab

Este notebook baixa a pasta do Google Drive, carrega uma voz treinada e abre uma caixa de texto. Digite uma frase e pressione **Enter** para gerar um arquivo de audio. Pressione **Esc** para desativar a caixa.

Formatos suportados automaticamente:
- Coqui/YourTTS/VITS: `config.json` + arquivo `.pth`
- Piper: arquivo `.onnx` + arquivo `.json` do modelo

Se a sua pasta tiver outro formato, rode as celulas de verificacao e ajuste os caminhos indicados.

In [ ]:
# 1) Instalar bibliotecas e dependencias do sistema
!apt-get -qq update
!apt-get -qq install -y espeak-ng ffmpeg
!python -m pip install -q -U pip
!python -m pip install -q -U gdown gradio pydub soundfile

# Coqui TTS mudou de pacote ao longo do tempo. O pacote atual preferencial e coqui-tts.
# Se ele falhar no seu runtime, o fallback tenta o pacote antigo TTS.
!python -m pip install -q -U coqui-tts || python -m pip install -q TTS

# Necessario somente se sua voz treinada estiver em formato Piper (.onnx).
!python -m pip install -q -U piper-tts || true

In [ ]:
# 2) Baixar a pasta publica do Google Drive
from pathlib import Path

DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/13uBrrPLx--PlHho0fcz5xW8eYehimRcC?usp=sharing"
MODEL_ROOT = Path("/content/voz_treinada")
MODEL_ROOT.mkdir(parents=True, exist_ok=True)

!gdown --folder "{DRIVE_FOLDER_URL}" -O "{MODEL_ROOT}" --remaining-ok

print("Arquivos encontrados na pasta baixada:")
for path in sorted(MODEL_ROOT.rglob("*")):
    if path.is_file():
        size_mb = path.stat().st_size / 1024 / 1024
        print(f"{path} ({size_mb:.1f} MB)")

In [ ]:
# 3) Detectar automaticamente o tipo de modelo
from pathlib import Path

MODEL_ROOT = Path("/content/voz_treinada")

def find_coqui_model(root: Path):
    configs = sorted(root.rglob("config.json"))
    pths = sorted(
        [p for p in root.rglob("*.pth") if not any(x in p.name.lower() for x in ["optimizer", "dvae", "speaker"])]
    )
    preferred = [p for p in pths if any(x in p.name.lower() for x in ["best", "model", "checkpoint"])] or pths
    if configs and preferred:
        return preferred[0], configs[0]
    return None, None

def find_piper_model(root: Path):
    onnx_files = sorted(root.rglob("*.onnx"))
    if not onnx_files:
        return None
    return onnx_files[0]

COQUI_MODEL_PATH, COQUI_CONFIG_PATH = find_coqui_model(MODEL_ROOT)
PIPER_MODEL_PATH = find_piper_model(MODEL_ROOT)

if COQUI_MODEL_PATH and COQUI_CONFIG_PATH:
    ENGINE = "coqui"
    print("Modelo Coqui detectado:")
    print("MODEL_PATH =", COQUI_MODEL_PATH)
    print("CONFIG_PATH =", COQUI_CONFIG_PATH)
elif PIPER_MODEL_PATH:
    ENGINE = "piper"
    print("Modelo Piper detectado:")
    print("MODEL_PATH =", PIPER_MODEL_PATH)
else:
    raise FileNotFoundError(
        "Nao encontrei um modelo suportado. Esperado: config.json + .pth para Coqui, ou .onnx para Piper. "
        "Confira a lista de arquivos da celula anterior e ajuste os caminhos manualmente."
    )

In [ ]:
# 4) Carregar a voz treinada
import os
import subprocess
from pathlib import Path

import torch
from pydub import AudioSegment

OUTPUT_DIR = Path("/content/audios_gerados")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

tts = None

if ENGINE == "coqui":
    from TTS.api import TTS

    use_gpu = torch.cuda.is_available()
    tts = TTS(
        model_path=str(COQUI_MODEL_PATH),
        config_path=str(COQUI_CONFIG_PATH),
        progress_bar=True,
        gpu=use_gpu,
    )
    print("Coqui carregado. GPU:", use_gpu)
    print("Speakers:", getattr(tts, "speakers", None))
    print("Languages:", getattr(tts, "languages", None))
else:
    print("Piper pronto para inferencia via CLI:", PIPER_MODEL_PATH)

def choose_first(values, preferred=None):
    if not values:
        return None
    preferred = preferred or []
    lowered = {str(v).lower(): v for v in values}
    for item in preferred:
        if item.lower() in lowered:
            return lowered[item.lower()]
    return values[0]

def wav_to_mp3(wav_path: Path) -> Path:
    mp3_path = wav_path.with_suffix(".mp3")
    AudioSegment.from_wav(wav_path).export(mp3_path, format="mp3")
    return mp3_path

def synthesize(text: str, output_format: str = "wav") -> str:
    text = (text or "").strip()
    if not text:
        raise ValueError("Digite algum texto antes de gerar o audio.")

    safe_id = f"audio_{len(list(OUTPUT_DIR.glob('audio_*.wav'))) + 1:04d}"
    wav_path = OUTPUT_DIR / f"{safe_id}.wav"

    if ENGINE == "coqui":
        kwargs = {"text": text, "file_path": str(wav_path)}
        speakers = getattr(tts, "speakers", None)
        languages = getattr(tts, "languages", None)
        speaker = choose_first(speakers)
        language = choose_first(languages, preferred=["pt-br", "pt", "portuguese", "portugues"])
        if speaker:
            kwargs["speaker"] = speaker
        if language:
            kwargs["language"] = language
        tts.tts_to_file(**kwargs)
    else:
        command = ["python", "-m", "piper", "-m", str(PIPER_MODEL_PATH), "-f", str(wav_path)]
        subprocess.run(command, input=text, text=True, check=True)

    if output_format == "mp3":
        return str(wav_to_mp3(wav_path))
    return str(wav_path)

print("Funcao synthesize pronta. Teste rapido:")
test_file = synthesize("Ola, este e um teste da voz treinada.", "wav")
print(test_file)

In [ ]:
# 5) Abrir app: digite e pressione Enter para gerar audio. Pressione Esc para sair da caixa.
import gradio as gr

app_state = {"active": True}

def generate_from_box(text, output_format):
    if not app_state["active"]:
        return None, None, "Caixa encerrada. Rode esta celula novamente para reabrir."
    try:
        audio_path = synthesize(text, output_format)
        return audio_path, audio_path, f"Audio gerado: {audio_path}"
    except Exception as exc:
        return None, None, f"Erro: {exc}"

def stop_box():
    app_state["active"] = False
    return gr.update(value="", interactive=False, placeholder="Caixa encerrada"), "Caixa encerrada."

ESC_JS = """
() => {
  document.addEventListener('keydown', (event) => {
    if (event.key === 'Escape') {
      const stopButton = document.querySelector('#stop_box_button');
      if (stopButton) stopButton.click();
    }
  });
}
"""

with gr.Blocks(title="Teste da voz treinada", js=ESC_JS) as demo:
    gr.Markdown("## Teste da voz treinada\nDigite uma frase e pressione Enter. Cada envio gera um novo audio para ouvir e baixar. Pressione Esc para encerrar a caixa.")
    text_box = gr.Textbox(
        label="Texto",
        placeholder="Digite aqui e pressione Enter...",
        lines=1,
        autofocus=True,
    )
    output_format = gr.Radio(["wav", "mp3"], value="wav", label="Formato")
    audio = gr.Audio(label="Audio gerado", type="filepath")
    download = gr.File(label="Baixar arquivo")
    status = gr.Textbox(label="Status", interactive=False)
    stop_button = gr.Button("Encerrar", elem_id="stop_box_button", visible=False)

    text_box.submit(
        fn=generate_from_box,
        inputs=[text_box, output_format],
        outputs=[audio, download, status],
    ).then(lambda: "", outputs=text_box)
    stop_button.click(fn=stop_box, inputs=None, outputs=[text_box, status])

demo.launch(share=True, debug=True)